# Lab1: Load a pretrained Chinese word embedding model and run basic semantic checks

- Load the embedding model: https://github.com/Embedding/Chinese-Word-Vectors

- Load the model: model = KeyedVectors.load_word2vec_format(model_path)
- Find most similar: model.most_similar()
- Similarity calculation: model.similarity()

In [14]:
from gensim.models import KeyedVectors
from pathlib import Path

In [16]:
model_path = Path('word_embedding/sgns.weibo.word.bz2')
# Load the pre-trained word2vec model
model = KeyedVectors.load_word2vec_format(model_path)


In [ ]:
print(model.vector_size) # print the size of the word vectors
# print(model['地铁']) # print the vector for the word '地铁'
print(model.most_similar('地铁', topn=10)) # print the top 10 most similar words to '地铁'
print(model.similarity('地铁', '福州')) # print the similarity between '地铁' and '福州'

300
[('二号线', 0.6998022198677063), ('四号线', 0.6872349381446838), ('北京地铁', 0.6863653659820557), ('一号线', 0.6666116714477539), ('地铁站', 0.659015417098999), ('公交', 0.654582142829895), ('五号线', 0.653891384601593), ('坐地铁', 0.6435028314590454), ('军博', 0.6391095519065857), ('八通线', 0.632388174533844)]
0.4049673


In [ ]:
print(model.most_similar(positive=['地铁', '飞机'], negative=['饭店'], topn=3)) # print the top 10 most similar words to '地铁' and '飞机' but not '饭店'

[('起飞时', 0.5109537839889526), ('快轨', 0.501717209815979), ('北京地铁', 0.5009052753448486)]


# Lab2: Train a new word2vec model on your own data
- Provide a training set with token. (this can also be done with jieba when the language is Chinese)
- Use Word2Vec model from gensim.models library. Key parameters include vector_size, windows, workers, sg (skip-gram or not)
- The trained model can be stored through 'save_word2vec_format'

In [17]:
# Provide the training set:
sentences = [['我', '每天','乘坐', '地铁', '上班'], ['我','每天', '乘坐', '公交', '上班']]

In [18]:
#Training a new word2vec model
from gensim.models import Word2Vec
model = Word2Vec(
    sentences=sentences, #tokenized sentences
    vector_size=100, # size of the word vectors
    window=5, # maximum distance between the current and predicted word within a sentence
    min_count=1, # ignores all words with total frequency lower than this
    workers=4, # number of threads to use while training
    sg=1  # use skip-gram; if 0, CBOW is used
)

In [19]:
# to save the model
model.wv.save_word2vec_format('word_embedding/my_word2vec_model.kv') #save the model in Gensim KeyedVectors format
my_model = KeyedVectors.load_word2vec_format('word_embedding/my_word2vec_model.kv') #load the model

## Lab2.1 Train a w2v model with online data

In [1]:
import jieba
from gensim.models import Word2Vec, KeyedVectors
import pandas as pd


/Users/yanghanghuang/Desktop/GitHub/NLP_Lab/.venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [11]:
input_file = '/Users/yanghanghuang/Desktop/GitHub/NLP_Lab/word2vec/word_embedding/online_shopping_10_cats.csv'
df = pd.read_csv(input_file, encoding='utf-8', usecols=['review'])

In [12]:
sentences = [
    [token for token in jieba.lcut(review) if token.strip() != ''] 
    for review in df['review'].dropna()
    ]

In [13]:
model = Word2Vec(
    sentences=sentences, #tokenized sentences
    vector_size=100, # size of the word vectors
    window=5, # maximum distance between the current and predicted word within a sentence
    min_count=1, # ignores all words with total frequency lower than this
    workers=4, # number of threads to use while training
    sg=1  # use skip-gram; if 0, CBOW is used
)

model.wv.save_word2vec_format('word_embedding/my_word2vec_model_v1.kv') #save the model in Gensim KeyedVectors format

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [14]:
my_model = KeyedVectors.load_word2vec_format('word_embedding/my_word2vec_model_v1.kv') #load the model
print(my_model.vector_size) # print the size of the word vectors

100


# Lab3: Apply the Trained W2V into NLP
- In many NLP NN, the first trainable component is an embedding layer that maps *Token ID* to *Dense Vectors*
- Typical NLP pipeline: Raw text -> tokenizer -> token IDs -> embedding vectors -> NN -> prediction
- To use the pretrained model in the NLP embedding layer, in PyTorch, the key API is nn.Embedding.from_pretrained.
  - 'embedding_layer = nn.Embedding.from_pretrained(
      embedding_matrix, # the LUT with a shape of (num_embeddigns,embedding_dim)
      freeze=False  # freeze the embedding_matrix?
  )'
  - embedding_matrix is basically a lookup table used by the embedding layer. Receive the token IDs -> LUT -> Output vectors
  

In [1]:
import torch
import torch.nn as nn
from gensim.models import KeyedVectors

In [2]:
# Load W2V model
word_vectors = KeyedVectors.load_word2vec_format('word_embedding/my_word2vec_model_v1.kv')
# Build embedding matrix (number of embeddings, embedding dimension), which is basically a lookup table.

embedding_dim = word_vectors.vector_size
number_of_embeddings = len(word_vectors.key_to_index) + 2  # .key_to_index is a dictionary; +2 for padding and unknown index
embedding_matrix = torch.zeros(number_of_embeddings, embedding_dim)

# Reserve the first two rows for padding and unknown tokens
embedding_matrix[0] = torch.zeros(embedding_dim)  # padding index
embedding_matrix[1] = torch.zeros(embedding_dim)  # unknown index

for i, word in enumerate(word_vectors.key_to_index): # enumerate over the words in the vocabulary to build the embedding matrix
    embedding_matrix[i+2] = torch.tensor(word_vectors[word])

# Build embedding layer
embedding_layer = nn.Embedding.from_pretrained(embedding_matrix, freeze=False) # freeze=False means the embeddings will be updated during training

In [3]:
# Example usage of the embedding layer
input_words = ["我", "喜欢", "乘坐", "地铁"]
input_indices = torch.tensor([word_vectors.key_to_index[word] + 2 for word in input_words])

output_embeddings = embedding_layer(input_indices)
print(output_embeddings) # print the embeddings for the input words

tensor([[ 0.1427,  0.0626,  0.1096, -0.1545,  0.0930, -0.4988,  0.4933,  0.2532,
          0.0614, -0.1105,  0.3817, -0.2788, -0.1996, -0.0658, -0.0157, -0.3887,
          0.1583, -0.3387, -0.2322, -0.8477,  0.1875, -0.1572,  0.4438, -0.4020,
         -0.3234,  0.1443,  0.1739,  0.1084, -0.0826, -0.1990,  0.5144, -0.0055,
         -0.0174, -0.4264, -0.1731, -0.0685,  0.1638, -0.2086, -0.1381, -0.4684,
          0.0492, -0.2495, -0.0160,  0.0394,  0.1647,  0.0171,  0.1727, -0.1405,
          0.2794,  0.2004, -0.2137, -0.2311,  0.5474, -0.0511, -0.3338,  0.2245,
         -0.0636,  0.1431, -0.2125,  0.2369,  0.4253, -0.2975, -0.3775, -0.1590,
          0.2820,  0.4301, -0.0126,  0.2962, -0.3567, -0.0092,  0.0642, -0.3871,
          0.2617, -0.5466,  0.0580,  0.2876,  0.1921,  0.1194, -0.1982,  0.0668,
         -0.3152,  0.2302, -0.2445,  0.1218, -0.3078, -0.1846,  0.3472,  0.4164,
          0.2568, -0.2220,  0.0152, -0.2765, -0.4916,  0.3033,  0.6189,  0.0280,
          0.2284, -0.3590, -